# SmartFeed pipeline

Runs every module that has an input, then returns **feed / dilute / reject**.

| Input | Module | Notebook |
|---|---|---|
| Ingredient name + form | Nutrition lookup / ExtraTrees | `notebooks/ml/nutrition_estimator.ipynb` |
| Moisture % | As-fed convert | `notebooks/moisture/asfed_moisture.ipynb` |
| pH + moisture | Flieg | `notebooks/silage/flieg_silage.ipynb` |
| 4-DMAB yellow % | Urea | `notebooks/urea/urea_4dmab.ipynb` |
| Photo / mould flag | CNN (HSV side-check) | `notebooks/cv/train_mould_cnn.py` |
| Ingredient | AFB1 risk | `notebooks/aflatoxin/afb1_risk.ipynb` |
| Jar grit / AIA % | Sand / silica | — |
| 256-band NIR | Forage PLS (Brazil only) | `notebooks/nir/nir_forage_estimator.ipynb` |
| 18-ch AS7265x | Refused until Indian pairs exist | — |

Python API: `from smartfeed import assess`


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("../..").resolve()
if not (ROOT / "smartfeed").exists():
    ROOT = Path(".").resolve()
    while ROOT != ROOT.parent and not (ROOT / "smartfeed").exists():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "processed"
print("ROOT", ROOT)
print("DATA", DATA)


ROOT /Users/sayangarai/Documents/Programming/Projects/SIH_PS_260111
DATA /Users/sayangarai/Documents/Programming/Projects/SIH_PS_260111/data/processed


In [2]:
from smartfeed import assess

cases = [
    assess("mustard cake", moisture_pct=10),
    assess("cattle feed", form="compounded", moisture_pct=16),
    assess("maize", form="silage", moisture_pct=70, ph=3.9),
    assess("wheat silage", form="silage", moisture_pct=78, ph=5.6),
    assess("groundnut cake", moisture_pct=11, urea_yellow_area_pct=62, visible_mould=True),
    assess("unknown green cake"),
    assess("wheat bran", nir_absorbance=[0.1] * 18),
]
for c in cases:
    print("=" * 64)
    print(c["ingredient"], "->", c["decision"])
    nut = c["modules"]["nutrition"]
    print("  nutrition", nut.get("method"), nut.get("canonical"), nut.get("feed_class"))
    if c["modules"]["nir"].get("note"):
        print("  nir", c["modules"]["nir"]["note"][:90])


mustard cake -> {'action': 'feed', 'reasons': ['no reject/dilute trigger from sensors + tables']}
  nutrition indian_table_lookup mustard cake oilcakes
  nir No spectrum.
cattle feed -> {'action': 'dilute', 'reasons': ['compounded-feed moisture above BIS 11%']}
  nutrition bis_standard_only compounded feed None
  nir No spectrum.
maize -> {'action': 'feed', 'reasons': ['no reject/dilute trigger from sensors + tables']}
  nutrition indian_table_lookup maize silage silage
  nir No spectrum.
wheat silage -> {'action': 'dilute', 'reasons': ['silage Flieg 25.0 moderate']}
  nutrition indian_table_lookup wheat silage silage
  nir No spectrum.
groundnut cake -> {'action': 'reject', 'reasons': ['4-DMAB yellow — suspect urea', 'visible mould', 'high-AF-risk ingredient plus mould']}
  nutrition indian_table_lookup groundnut cake oilcakes
  nir No spectrum.
unknown green cake -> {'action': 'feed', 'reasons': ['no reject/dilute trigger from sensors + tables']}
  nutrition best_cv_name_model unknow

In [3]:
import json
from pathlib import Path

demo = assess(
    "groundnut cake",
    moisture_pct=12,
    urea_yellow_area_pct=8,
    visible_mould=False,
)
art = Path("artifacts")
art.mkdir(exist_ok=True)
(art / "metrics.json").write_text(
    json.dumps(
        {
            "api": "smartfeed.assess",
            "modules": ["nutrition", "moisture", "silage", "urea", "cv", "aflatoxin", "sand", "nir"],
            "actions": ["feed", "dilute", "reject"],
        },
        indent=2,
    )
)
print(json.dumps(demo["decision"], indent=2))


{
  "action": "dilute",
  "reasons": [
    "4-DMAB yellow \u2014 suspect urea"
  ]
}
